# 06 - Stationary Distributions and Fokker-Planck Equation

This notebook explores the stationary distributions of SGD dynamics using the Fokker-Planck equation framework.

**Converted from:** `current_and_stationary.nb` (Mathematica)

## Contents:
1. Fokker-Planck equation for SGD
2. Stationary distribution computation
3. Current flow analysis
4. Equilibrium probability distributions
5. Relationship to Langevin dynamics

## Background

The Fokker-Planck equation describes the time evolution of the probability density of SGD parameters. At equilibrium, the stationary distribution provides insights into where SGD spends most of its time in parameter space.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from scipy import integrate
from scipy.ndimage import gaussian_filter

# Add utils to path
sys.path.insert(0, str(Path.cwd().parent / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import SmoothNonlinearLoss, generate_noisy_data
from visualization import plot_loss_landscape

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. Fokker-Planck Equation Framework

The Fokker-Planck equation for SGD in continuous time is:

$$\frac{\partial p(\theta, t)}{\partial t} = -\nabla \cdot \mathbf{J}(\theta, t)$$

where the probability current is:

$$\mathbf{J}(\theta, t) = -\nabla U(\theta) p(\theta, t) - D \nabla p(\theta, t)$$

Here:
- $U(\theta)$ is the loss function (potential)
- $D = \eta \sigma^2 / 2$ is the diffusion coefficient
- $\eta$ is the learning rate
- $\sigma^2$ is the gradient noise variance

In [ ]:
# Generate data for loss function
np.random.seed(42)

x_data, y_data = generate_noisy_data(
    x_range=(-3, 3),
    n_points=20,
    n_samples_per_point=5,
    noise_std=0.5,
    p=1.0,
    random_state=42
)

# Create loss function
loss_obj = SmoothNonlinearLoss(p=1.0)

print(f"Generated {len(x_data)} data points for loss computation")

## 2. Stationary Distribution

At equilibrium ($\partial p / \partial t = 0$), the stationary distribution satisfies:

$$\nabla \cdot \mathbf{J}_{\text{stat}}(\theta) = 0$$

For the special case where the current is zero everywhere, the stationary distribution is:

$$p_{\text{stat}}(\theta) \propto \exp\left(-\frac{U(\theta)}{D}\right) = \exp\left(-\frac{L(\theta)}{\eta \sigma^2 / 2}\right)$$

This is a Gibbs distribution with "effective temperature" $T_{\text{eff}} = \eta \sigma^2 / 2$.

In [ ]:
def compute_stationary_distribution(loss_fn, x_data, y_data, param_range, 
                                   learning_rate, gradient_noise_var, n_points=100):
    """
    Compute the stationary distribution from the Fokker-Planck equation.
    """
    (a_min, a_max), (b_min, b_max) = param_range
    a_vals = np.linspace(a_min, a_max, n_points)
    b_vals = np.linspace(b_min, b_max, n_points)
    A, B = np.meshgrid(a_vals, b_vals)
    
    # Compute loss at each point
    L = np.zeros_like(A)
    for i in range(n_points):
        for j in range(n_points):
            params = np.array([A[i, j], B[i, j]])
            L[i, j] = loss_fn(params, x_data, y_data)
    
    # Effective temperature
    D = learning_rate * gradient_noise_var / 2
    
    # Stationary distribution (Gibbs distribution)
    p_stat = np.exp(-L / D)
    
    # Normalize
    da = (a_max - a_min) / (n_points - 1)
    db = (b_max - b_min) / (n_points - 1)
    normalization = np.sum(p_stat) * da * db
    p_stat = p_stat / normalization
    
    return A, B, p_stat, L

# Compute stationary distribution
param_range = ((-1, 3), (-1, 3))
learning_rate = 0.05
gradient_noise_var = 0.1  # Estimated gradient noise variance

A, B, p_stat, L = compute_stationary_distribution(
    loss_obj, x_data, y_data, param_range, 
    learning_rate, gradient_noise_var, n_points=100
)

print(f"Effective temperature: {learning_rate * gradient_noise_var / 2:.4f}")
print(f"Stationary distribution computed on {100}x{100} grid")

In [ ]:
# Plot stationary distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Loss landscape
contourf1 = axes[0].contourf(A, B, L, levels=30, cmap='viridis')
axes[0].contour(A, B, L, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf1, ax=axes[0], label='Loss')
axes[0].set_xlabel('Parameter a', fontsize=12)
axes[0].set_ylabel('Parameter b', fontsize=12)
axes[0].set_title('Loss Landscape', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Right: Stationary distribution
contourf2 = axes[1].contourf(A, B, p_stat, levels=30, cmap='plasma')
axes[1].contour(A, B, p_stat, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf2, ax=axes[1], label='Probability Density')
axes[1].set_xlabel('Parameter a', fontsize=12)
axes[1].set_ylabel('Parameter b', fontsize=12)
axes[1].set_title('Stationary Distribution', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Stationary distribution concentrates in low-loss regions!")

## 3. Current Flow Analysis

The probability current reveals how probability flows in parameter space:

$$\mathbf{J}(\theta) = -\nabla U(\theta) p(\theta) - D \nabla p(\theta)$$

At equilibrium, we can have:
1. **Zero current everywhere**: $\mathbf{J} = 0$ (detailed balance)
2. **Non-zero current with zero divergence**: $\nabla \cdot \mathbf{J} = 0$ (steady-state circulation)

Let's visualize the current flow.

In [ ]:
def compute_current_flow(loss_fn, x_data, y_data, p_stat, A, B, 
                        learning_rate, gradient_noise_var):
    """
    Compute probability current flow.
    """
    n_points = A.shape[0]
    D = learning_rate * gradient_noise_var / 2
    
    # Compute gradients of loss and probability
    grad_L_a = np.zeros_like(A)
    grad_L_b = np.zeros_like(A)
    
    for i in range(n_points):
        for j in range(n_points):
            params = np.array([A[i, j], B[i, j]])
            grad = loss_fn.gradient(params, x_data, y_data)
            grad_L_a[i, j] = grad[0]
            grad_L_b[i, j] = grad[1]
    
    # Numerical gradient of probability
    grad_p_a, grad_p_b = np.gradient(p_stat, A[0, 1] - A[0, 0], B[1, 0] - B[0, 0])
    
    # Current components
    J_a = -grad_L_a * p_stat - D * grad_p_a
    J_b = -grad_L_b * p_stat - D * grad_p_b
    
    return J_a, J_b

# Compute current
J_a, J_b = compute_current_flow(loss_obj, x_data, y_data, p_stat, A, B,
                                learning_rate, gradient_noise_var)

print("Probability current computed")
print(f"Max current magnitude: {np.max(np.sqrt(J_a**2 + J_b**2)):.4e}")

In [ ]:
# Plot current flow
fig, ax = plt.subplots(figsize=(12, 10))

# Background: stationary distribution
contourf = ax.contourf(A, B, p_stat, levels=20, cmap='plasma', alpha=0.5)
plt.colorbar(contourf, ax=ax, label='Stationary Probability Density')

# Overlay: current flow (subsampled for clarity)
skip = 5
ax.quiver(A[::skip, ::skip], B[::skip, ::skip], 
         J_a[::skip, ::skip], J_b[::skip, ::skip],
         scale=np.max(np.sqrt(J_a**2 + J_b**2)) * 20,
         color='white', alpha=0.7, width=0.003)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('Probability Current Flow on Stationary Distribution', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("White arrows show direction and magnitude of probability current")

## 4. Equilibrium Probability Distributions

Let's examine the marginal distributions along each parameter axis.

In [ ]:
# Compute marginal distributions
da = (A[0, 1] - A[0, 0])
db = (B[1, 0] - B[0, 0])

# Marginal for parameter a (integrate over b)
p_a = np.sum(p_stat, axis=0) * db
a_vals = A[0, :]

# Marginal for parameter b (integrate over a)
p_b = np.sum(p_stat, axis=1) * da
b_vals = B[:, 0]

# Plot marginals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(a_vals, p_a, 'b-', linewidth=2.5)
axes[0].fill_between(a_vals, p_a, alpha=0.3)
axes[0].set_xlabel('Parameter a', fontsize=12)
axes[0].set_ylabel('Probability Density', fontsize=12)
axes[0].set_title('Marginal Distribution for Parameter a', fontsize=14)
axes[0].grid(True, alpha=0.3)

axes[1].plot(b_vals, p_b, 'r-', linewidth=2.5)
axes[1].fill_between(b_vals, p_b, alpha=0.3, color='red')
axes[1].set_xlabel('Parameter b', fontsize=12)
axes[1].set_ylabel('Probability Density', fontsize=12)
axes[1].set_title('Marginal Distribution for Parameter b', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean of a: {np.sum(a_vals * p_a) * da:.3f}")
print(f"Mean of b: {np.sum(b_vals * p_b) * db:.3f}")

## 5. Relationship to Langevin Dynamics

SGD can be viewed as discretized Langevin dynamics:

$$d\theta_t = -\nabla U(\theta_t) dt + \sqrt{2D} dW_t$$

where $W_t$ is a Wiener process. The stationary distribution of this SDE is exactly the Gibbs distribution.

Let's verify this by running long SGD simulations and comparing the empirical distribution to the theoretical one.

In [ ]:
# Run long SGD simulation to sample from stationary distribution
sgd_config = SGDConfig(
    learning_rate=learning_rate,
    batch_size=10,
    n_iterations=50000,
    random_state=42
)

simulator = SGDSimulator(sgd_config)
initial_params = np.array([1.0, 1.0])

trajectory, iterations = simulator.run_trajectory(
    initial_params=initial_params,
    gradient_fn=loss_obj.gradient,
    x_data=x_data,
    y_data=y_data,
    save_every=10
)

print(f"Sampled {len(trajectory)} points from SGD trajectory")
print(f"Empirical mean: a={np.mean(trajectory[:, 0]):.3f}, b={np.mean(trajectory[:, 1]):.3f}")

In [ ]:
# Compare empirical and theoretical distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Theoretical stationary distribution
contourf1 = axes[0].contourf(A, B, p_stat, levels=30, cmap='plasma')
axes[0].contour(A, B, p_stat, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf1, ax=axes[0], label='Probability Density')
axes[0].set_xlabel('Parameter a', fontsize=12)
axes[0].set_ylabel('Parameter b', fontsize=12)
axes[0].set_title('Theoretical Stationary Distribution', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Right: Empirical distribution from SGD
hist, xedges, yedges = np.histogram2d(
    trajectory[:, 0], trajectory[:, 1], 
    bins=50, range=[[param_range[0][0], param_range[0][1]], 
                    [param_range[1][0], param_range[1][1]]],
    density=True
)

# Smooth the histogram
hist_smooth = gaussian_filter(hist.T, sigma=1.0)

X, Y = np.meshgrid(xedges[:-1], yedges[:-1])
contourf2 = axes[1].contourf(X, Y, hist_smooth, levels=30, cmap='plasma')
axes[1].contour(X, Y, hist_smooth, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf2, ax=axes[1], label='Probability Density')
axes[1].set_xlabel('Parameter a', fontsize=12)
axes[1].set_ylabel('Parameter b', fontsize=12)
axes[1].set_title('Empirical Distribution from SGD', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The empirical distribution closely matches the theoretical prediction!")

## Summary

In this notebook, we explored:

1. **Fokker-Planck Equation**: The fundamental equation governing the evolution of probability density in SGD
2. **Stationary Distribution**: Computed the equilibrium Gibbs distribution $p_{\text{stat}} \propto \exp(-L/T_{\text{eff}})$
3. **Current Flow**: Analyzed how probability flows in parameter space
4. **Marginal Distributions**: Examined the probability distribution along each parameter axis
5. **Langevin Connection**: Verified that long SGD runs sample from the theoretical stationary distribution

**Key Insights:**
- SGD at equilibrium follows a Gibbs distribution with effective temperature $T_{\text{eff}} = \eta \sigma^2 / 2$
- Higher learning rates or gradient noise lead to broader stationary distributions
- The stationary distribution concentrates probability in low-loss regions
- Probability current can reveal non-equilibrium dynamics and circulation patterns

**Next Steps:**
- Explore 2D Ito differential equations and stochastic trajectories
- Study phase space visualization of SGD dynamics